In [1]:
import pickle
import torch

# Load the pickle file
with open("features_test.pkl", "rb") as f:
    features_dict = pickle.load(f)

# Print keys and shapes
for key, tensor in features_dict.items():
    print(f"Key: {key} — Shape: {tensor.shape} — Dtype: {tensor.dtype}")


Key: RSI — Shape: torch.Size([416, 1024]) — Dtype: torch.float32
Key: SVI — Shape: torch.Size([416, 1024]) — Dtype: torch.float32
Key: UAV — Shape: torch.Size([416, 1024]) — Dtype: torch.float32
Key: VGI — Shape: torch.Size([416, 1024]) — Dtype: torch.float32


In [2]:
import torch
import pickle
import numpy as np
from collections import defaultdict

# Load features
with open("features_test.pkl", "rb") as f:
    features_dict = pickle.load(f)

modalities = list(features_dict.keys())
num_samples = next(iter(features_dict.values())).shape[0]
k_values = [1, 5]

def compute_recall_at_k(query_feats, target_feats, ks):
    # Compute cosine similarity matrix (dot product since features are normalized)
    sims = query_feats @ target_feats.T  # shape: (N, N)
    ranks = torch.argsort(sims, dim=1, descending=True)  # shape: (N, N)

    recalls = {}
    targets = torch.arange(query_feats.shape[0]).unsqueeze(1)  # shape: (N, 1)

    for k in ks:
        topk = ranks[:, :k]  # shape: (N, k)
        correct = (topk == targets).any(dim=1).float()  # shape: (N,)
        recalls[f"R@{k}"] = correct.mean().item()

    return recalls

# Compute recall between each modality pair (both directions)
results = defaultdict(dict)

for query_modality in modalities:
    for target_modality in modalities:
        if query_modality == target_modality:
            continue
        query_feats = features_dict[query_modality]
        target_feats = features_dict[target_modality]
        recalls = compute_recall_at_k(query_feats, target_feats, k_values)
        results[f"{query_modality}→{target_modality}"] = recalls

# Print results
for pair, metrics in results.items():
    print(f"{pair}: " + ", ".join([f"{k}: {v:.4f}" for k, v in metrics.items()]))


RSI→SVI: R@1: 0.6659, R@5: 0.8654
RSI→UAV: R@1: 0.7837, R@5: 0.9087
RSI→VGI: R@1: 0.3029, R@5: 0.5889
SVI→RSI: R@1: 0.6779, R@5: 0.8750
SVI→UAV: R@1: 0.6635, R@5: 0.9183
SVI→VGI: R@1: 0.3990, R@5: 0.7212
UAV→RSI: R@1: 0.7644, R@5: 0.9087
UAV→SVI: R@1: 0.6394, R@5: 0.9111
UAV→VGI: R@1: 0.2716, R@5: 0.6058
VGI→RSI: R@1: 0.3317, R@5: 0.6250
VGI→SVI: R@1: 0.3942, R@5: 0.7332
VGI→UAV: R@1: 0.2837, R@5: 0.6010


In [4]:
def anchor_based_rerank(vgi, rsi, uav, svi, topk=(1, 5)):
    # Step 1: Build anchor features for each sample
    anchor = (rsi + uav + svi) / 3.0  # shape: [N, D]
    anchor = anchor / anchor.norm(dim=1, keepdim=True)  # normalize

    # Step 2: Compute similarity between VGI and anchors
    sim_vgi_anchor = vgi @ anchor.T  # [N, N]

    # Step 3: Re-rank RSI using anchor similarities
    ranks = torch.argsort(sim_vgi_anchor, dim=1, descending=True)
    targets = torch.arange(vgi.shape[0]).unsqueeze(1)

    # Step 4: Compute recall
    recalls = {}
    for k in topk:
        correct = (ranks[:, :k] == targets).any(dim=1).float().mean().item()
        recalls[f"R@{k}"] = correct

    return recalls
vgi = features_dict["VGI"]
rsi = features_dict["RSI"]
uav = features_dict["UAV"]
svi = features_dict["SVI"]

recalls = anchor_based_rerank(vgi, rsi, uav, svi)
print(f"Anchor-based Re-ranked VGI→RSI: R@1: {recalls['R@1']:.4f}, R@5: {recalls['R@5']:.4f}")


Anchor-based Re-ranked VGI→RSI: R@1: 0.4736, R@5: 0.7548
